# 3CGL Slope 特徵重要性與離散特徵分析（Google Colab 專用）

本筆記本直接從 **Google Drive** 讀取訓練資料，並從 **GitHub** (`NickJJChen/CGL-python-project`) 拉取最新程式碼執行，避免將大量資料下載至本機造成傳輸緩慢與記憶體溢位。

### 使用前建議設定
1. 點選上方選單 **執行階段 (Runtime)** → **變更執行階段類型 (Change runtime type)**。
2. 硬體加速器建議選擇 **GPU**（例如 T4 / L4），並勾選 **高 RAM (High-RAM)** 執行階段。
3. 依序執行下方儲存格即可。

In [ ]:
#@title 1. 自動掛載 Google Drive 與檢查硬體資源 (GPU / RAM)
import os
import subprocess
from google.colab import drive

drive.mount('/content/drive')

# 檢查 GPU 狀態
gpu_check = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
HAS_GPU = (gpu_check.returncode == 0)
if HAS_GPU:
    print("✅ 偵測到可用 GPU：")
    print(gpu_check.stdout.splitlines()[8] if len(gpu_check.stdout.splitlines()) > 8 else "GPU Ready")
else:
    print("⚠️ 未偵測到 GPU，將使用 CPU 執行。若需加速請切換執行階段類型為 GPU。")

# 檢查系統記憶體
with open("/proc/meminfo") as f:
    mem_lines = {line.split(":")[0]: line.split(":")[1].strip() for line in f if ":" in line}
mem_total_kb = int(mem_lines.get("MemTotal", "0 kB").split()[0])
print(f"💾 系統可用總記憶體: {mem_total_kb / (1024 ** 2):.1f} GB")

In [ ]:
#@title 2. 從 GitHub 下載 / 更新程式碼
import os
import subprocess

GITHUB_REPO = "https://github.com/NickJJChen/CGL-python-project.git"
REPO_DIR = "/content/CGL-python-project"

if os.path.exists(os.path.join(REPO_DIR, ".git")):
    print(f"🔄 更新既有專案：{REPO_DIR}")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
else:
    print(f"📥 Clone 專案：{GITHUB_REPO}")
    subprocess.run(["git", "clone", GITHUB_REPO, REPO_DIR], check=True)

PROJECT_DIR = os.path.join(REPO_DIR, "03 3CGL model_training")
print(f"📂 專案腳本目錄：{PROJECT_DIR}")

In [ ]:
#@title 3. 訓練參數設定表單
SCRIPT_NAME = "01_01.py"  #@param ["01.py", "01_01.py"]
DATA_PATH = "/content/drive/MyDrive/00 Antigravity/01 training data/3CGLdata"  #@param {type:"string"}
TITLE_PATH = "/content/drive/MyDrive/00 Antigravity/01 training data/3CGLdata/title01.txt"  #@param {type:"string"}
OUTPUT_DIR = "/content/drive/MyDrive/00 Antigravity/01 training data/3CGLdata/outputs"  #@param {type:"string"}
USE_GPU = True  #@param {type:"boolean"}
MONTHS = ""  #@param {type:"string"}
TREES_PER_MONTH = 20  #@param {type:"integer"}
CORR_SAMPLE_PER_MONTH = 30000  #@param {type:"integer"}

import os

# 檢查資料路徑；若頂層無 YYYYMMDD 但存在 data_new 子資料夾，自動切換
if not os.path.isdir(DATA_PATH):
    raise FileNotFoundError(f"找不到資料目錄：{DATA_PATH}，請確認 Google Drive 路徑是否正確。")

has_ym_dirs = any(
    os.path.isdir(os.path.join(DATA_PATH, e)) and len(e) >= 6 and e[:6].isdigit()
    for e in os.listdir(DATA_PATH)
)
if not has_ym_dirs and os.path.isdir(os.path.join(DATA_PATH, "data_new")):
    DATA_PATH = os.path.join(DATA_PATH, "data_new")
    print(f"ℹ️ 自動切換至子目錄：{DATA_PATH}")

# 若 Drive 路徑找不到 title01.txt，自動 fallback 到 GitHub repo 內的 title01.txt
if not os.path.isfile(TITLE_PATH):
    repo_title = os.path.join(PROJECT_DIR, "title01.txt")
    if os.path.isfile(repo_title):
        print(f"ℹ️ Drive 未找到 {TITLE_PATH}，改用專案內欄位檔：{repo_title}")
        TITLE_PATH = repo_title
    else:
        raise FileNotFoundError(f"找不到欄位定義檔：{TITLE_PATH}")

DEVICE = "cuda" if (USE_GPU and HAS_GPU) else "cpu"
if USE_GPU and not HAS_GPU:
    print("⚠️ 已勾選 USE_GPU 但環境無 GPU，自動改用 CPU。")

print("=" * 50)
print(f"執行腳本：{SCRIPT_NAME}")
print(f"資料路徑：{DATA_PATH}")
print(f"欄位檔案：{TITLE_PATH}")
print(f"輸出目錄：{OUTPUT_DIR}")
print(f"運算裝置：{DEVICE}")
print(f"指定月份：{MONTHS if MONTHS.strip() else '全部月份'}")
print(f"每月樹數：{TREES_PER_MONTH} | 每月抽樣：{CORR_SAMPLE_PER_MONTH:,}")
print("=" * 50)

In [ ]:
#@title 4. 執行模型訓練與特徵分析
import os
import subprocess
import sys

script_path = os.path.join(PROJECT_DIR, SCRIPT_NAME)
cmd = [
    sys.executable,
    "-u",
    script_path,
    "--data-path", DATA_PATH,
    "--title-path", TITLE_PATH,
    "--output-dir", OUTPUT_DIR,
    "--trees-per-month", str(TREES_PER_MONTH),
    "--corr-sample-per-month", str(CORR_SAMPLE_PER_MONTH),
    "--device", DEVICE,
]

if MONTHS.strip():
    cmd.extend(["--months", *MONTHS.strip().split()])

print("執行指令:", " ".join(f'"{c}"' if " " in c else c for c in cmd))
print("-" * 50)

process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end="")

ret = process.wait()
if ret != 0:
    raise RuntimeError(f"執行失敗，結束代碼：{ret}")
print("\n✅ 執行完成！結果已儲存至：", OUTPUT_DIR)

In [ ]:
#@title 5. 預覽輸出圖表與 CSV 結果
import glob
import os
import pandas as pd
from IPython.display import Image, display

csv_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.csv")))
for csv_file in csv_files:
    print(f"\n📄 {os.path.basename(csv_file)}")
    display(pd.read_csv(csv_file).head(15))

png_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.png")))
for png_file in png_files:
    print(f"\n🖼️ {os.path.basename(png_file)}")
    display(Image(filename=png_file))